[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adan-rs/amd/blob/main/notebooks/17_RLM_modelos.ipynb)

# Selección de modelos de regresión

*¿Para qué se utiliza?*
Cuando se dispone de muchas variables independientes candidatas, incluirlas todas en el modelo rara vez es la mejor opción: aumenta el riesgo de sobreajuste, dificulta la interpretación y puede generar coeficientes inestables por multicolinealidad. La selección de modelos busca un balance entre ajuste (qué tanto explica el modelo) y simplicidad (cuántas variables usa).

Ejemplos de uso en negocios:
- Un área de Recursos Humanos tiene 20 variables candidatas para explicar la rotación de personal y necesita quedarse con las que realmente importan para una intervención.
- Un equipo de marketing mix mide el gasto en múltiples canales (muy correlacionados entre sí) y quiere saber cuáles tienen un efecto propio sobre las ventas.
- Un analista financiero construye un modelo de riesgo con decenas de indicadores económicos y necesita un modelo parsimonioso para reportarlo a un comité.

*Métodos disponibles*
- **Por significancia estadística**: eliminación hacia atrás (backward), hacia adelante (forward) o por pasos (stepwise), basadas en el p-valor de cada coeficiente.
- **Por criterios de información**: comparar modelos con AIC o BIC, que penalizan la complejidad (número de variables) además del ajuste.
- **Por regularización**: Ridge y Lasso, que penalizan la magnitud de los coeficientes durante la propia estimación del modelo, en lugar de decidir la inclusión/exclusión de variables después de estimarlo.

*Cuidado clave*: ningún método de selección garantiza encontrar "el" modelo verdadero: son heurísticas que pueden ser inestables y sensibles a la muestra utilizada. Deben complementarse siempre con el criterio del analista sobre qué variables tienen sentido de negocio incluir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV, Lasso, lasso_path

## Preparación de los datos

In [ ]:
df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/burritos.xlsx')

In [ ]:
df.info()

In [ ]:
df.head(3)

In [ ]:
df.dropna(inplace=True)
df.info()

## 1. Selección por significancia estadística: eliminación hacia atrás

El método de eliminación hacia atrás (backward elimination) comienza con un modelo que incluye todas las variables independientes y, de manera iterativa, elimina la variable menos significativa (usualmente la que tiene el mayor valor de p superior a un umbral, como 0.05). Este proceso se repite hasta que todas las variables restantes son estadísticamente significativas.

In [ ]:
var_ind = ['Cost', 'Hunger', 'Tortilla', 'Temp', 'Meat',
           'Fillings', 'Meat:filling', 'Uniformity', 'Salsa',
           'Synergy', 'Wrap']
X = df[var_ind]
X = sm.add_constant(X)
y = df['overall']

In [ ]:
results = sm.OLS(y, X).fit()
results.summary()

¿Cuál variable deberíamos eliminar? ¿Qué criterios podríamos considerar?

In [ ]:
def backward_elimination(X, y, significance_level=0.05):
    while True:
        model = sm.OLS(y, X).fit()
        pvalues = model.pvalues.drop('const')  # Excluir la constante
        if pvalues.max() > significance_level:
            excl_var = pvalues.idxmax()
            print(f"Eliminando {excl_var} con p-valor {pvalues[excl_var]:.4f}")
            X = X.drop(columns=[excl_var])
        else:
            break
    print(f"\nVariables conservadas: {list(X.columns.drop('const'))}")
    return model

In [ ]:
model_backward = backward_elimination(X, y)

In [ ]:
model_backward.summary()

**Ejemplo de reporte de resultados**:
>"Se aplicó eliminación hacia atrás sobre un modelo con 11 variables candidatas para explicar la calificación general de un burrito. El proceso eliminó 4 variables (Hunger, Tortilla, Salsa y Cost) y conservó 7. El modelo reducido explicó prácticamente la misma varianza que el modelo completo (R² ajustada de 0.798 en ambos casos)"

## 2. Selección por regularización: Ridge y Lasso

*¿Para qué se utiliza?*
Ridge y Lasso son técnicas de regresión regularizada: en lugar de estimar los coeficientes solo minimizando el error, agregan una penalización sobre la magnitud de los coeficientes. Esto reduce el sobreajuste y mejora la estabilidad del modelo, especialmente cuando hay variables muy correlacionadas entre sí.

- **Ridge (penalización L2)**: reduce ("encoge") los coeficientes hacia cero, pero rara vez los deja exactamente en cero. Es útil cuando se quiere conservar todas las variables pero controlar su magnitud.
- **Lasso (penalización L1)**: puede llevar coeficientes exactamente a cero, lo que lo convierte en un método de selección automática de variables además de un método de estimación.

En ambos casos, la fuerza de la penalización se controla con un parámetro (α o λ, según la fuente): a mayor penalización, coeficientes más pequeños (y en Lasso, más variables eliminadas).

*Requisito clave*: a diferencia de la regresión por mínimos cuadrados, Ridge y Lasso son sensibles a la escala de las variables, por lo que **siempre deben estandarizarse** las variables independientes antes de ajustarlas (media 0, desviación estándar 1).

*¿Cómo elegir la penalización?* En vez de probar valores de forma manual, se utiliza validación cruzada (`RidgeCV`, `LassoCV` en scikit-learn), que prueba distintos niveles de penalización y elige el que mejor generaliza en subconjuntos de los datos no usados para el ajuste.

In [ ]:
scaler = StandardScaler()
X_std = scaler.fit_transform(df[var_ind])

In [ ]:
ridge = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5).fit(X_std, y)
lasso = LassoCV(cv=5, random_state=42).fit(X_std, y)

print(f'Ridge: alpha elegido por CV = {ridge.alpha_:.4f}, R2 = {ridge.score(X_std, y):.3f}')
print(f'Lasso: alpha elegido por CV = {lasso.alpha_:.4f}, R2 = {lasso.score(X_std, y):.3f}')

In [ ]:
coeficientes = pd.DataFrame({
    'Backward': [results.params.get(v, 0) if v in model_backward.model.exog_names else 0 for v in var_ind],
    'Ridge (CV)': ridge.coef_,
    'Lasso (CV)': lasso.coef_
}, index=var_ind)
coeficientes.round(3)

En este caso, ni Ridge ni Lasso eliminaron variables con el nivel de penalización que sugiere la validación cruzada. El resultado indica que, dentro de este conjunto de datos, las 11 variables aportan algo de señal genuina y una penalización fuerte no mejora la predicción fuera de muestra. Esto **no** significa que Lasso nunca elimine variables: para ilustrar esa propiedad, veamos qué ocurre si aumentamos artificialmente la penalización más allá del óptimo de validación cruzada.

In [ ]:
alphas, coefs, _ = lasso_path(X_std, y, alphas=np.logspace(-3, 0, 60))

plt.figure(figsize=(7, 5))
for i, var in enumerate(var_ind):
    plt.plot(alphas, coefs[i], label=var)
plt.axvline(lasso.alpha_, color='black', linestyle='--', label='alpha elegido por CV')
plt.xscale('log')
plt.xlabel('alpha (penalización)')
plt.ylabel('Coeficiente')
plt.title('Trayectoria de coeficientes de Lasso')
plt.legend(fontsize=7, loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

**Ejemplo de reporte de resultados**:
>"Se ajustaron modelos de regresión Ridge y Lasso, estandarizando previamente las variables independientes y seleccionando la penalización mediante validación cruzada de 5 particiones. El alpha óptimo resultó bajo tanto para Ridge (19.31) como para Lasso (≈0.002), por lo que ninguno de los dos métodos eliminó variables de forma automática con la penalización sugerida por la validación cruzada, sugiriendo que las 11 variables aportan señal genuina sobre la calificación del burrito."

## Comparación de métodos

In [ ]:
resumen = pd.DataFrame({
    'Método': ['Modelo completo', 'Backward elimination', 'Ridge (CV)', 'Lasso (CV)'],
    'Variables retenidas': [
        len(var_ind),
        len(model_backward.model.exog_names) - 1,
        len(var_ind),
        sum(abs(lasso.coef_) > 1e-8)
    ],
    'R2': [
        results.rsquared,
        model_backward.rsquared,
        ridge.score(X_std, y),
        lasso.score(X_std, y)
    ]
})
resumen

## Conclusiones
- El modelo completo no siempre es el mejor: un modelo más simple puede explicar prácticamente la misma varianza con menor riesgo de sobreajuste.
- Una R cuadrada alta no siempre implica un mejor modelo: hay que balancearla contra la complejidad (número de variables) con métricas como AIC, BIC o R² ajustada.
- La eliminación hacia atrás y la regularización pueden llegar a conclusiones distintas sobre qué variables "importan": la primera es más inestable con muestras pequeñas o variables correlacionadas; la segunda controla mejor la multicolinealidad, pero requiere elegir cuidadosamente el nivel de penalización.
- Ridge es preferible cuando se quiere conservar todas las variables pero controlar su magnitud; Lasso es preferible cuando el objetivo es simplificar el modelo eliminando variables poco informativas.
- Ningún método de selección sustituye el criterio de negocio: siempre vale la pena revisar si las variables retenidas (o eliminadas) tienen sentido para el problema que se está analizando.

**Documentación**
https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression-and-classification
https://scikit-learn.org/stable/modules/linear_model.html#lasso